# CIFAR-100 Extension — Adaptive Batch Size and Learning Rate Project

This notebook extends the existing CIFAR-10 study to CIFAR-100.

The purpose is to investigate whether the adaptive batch-size and learning-rate
strategies observed on CIFAR-10 also behave similarly on a more difficult
100-class image classification problem.

The original CIFAR-10 pipeline is kept unchanged. CIFAR-100 is treated as a
separate robustness / second-dataset study.

In [ ]:
!git status -sb

## Step 1 — Download and Inspect CIFAR-100

CIFAR-100 contains:

- 50,000 training images
- 10,000 official test images
- 100 classes
- 32 × 32 RGB images

Each training image has one class label.

Before building the CIFAR-100 training pipeline, we first inspect the dataset
and verify its size, number of classes, and class distribution.

In [ ]:
from torchvision import datasets
import numpy as np

# ---------------------------------------------------------
# Load the ORIGINAL CIFAR-100 training set
# No augmentation and no normalization yet.
# ---------------------------------------------------------

cifar100_train = datasets.CIFAR100(
    root="data/downloads",
    train=True,
    download=True
)

images = cifar100_train.data
labels = np.array(cifar100_train.targets)

print("Image array shape:", images.shape)
print("Number of labels:", len(labels))
print("Number of classes:", len(np.unique(labels)))

# ---------------------------------------------------------
# Verify class distribution
# ---------------------------------------------------------

class_counts = np.bincount(labels, minlength=100)

print("Samples per class:")
print(class_counts)

# ---------------------------------------------------------
# Calculate channel mean and standard deviation
#
# We process the data in chunks so we don't create one
# enormous floating-point copy of the entire dataset.
## Step 2 — Calculate CIFAR-100 Normalization Statistics

# Neural networks usually train more effectively when image values are normalized.

# For each RGB channel, we calculate:

# - Mean
# - Standard deviation

# These statistics are calculated directly from the CIFAR-100 training dataset
# rather than reusing the CIFAR-10 normalization values.

# The resulting values will later be used with `transforms.Normalize()`.

# ---------------------------------------------------------

channel_sum = np.zeros(3, dtype=np.float64)
channel_squared_sum = np.zeros(3, dtype=np.float64)

total_pixels = 0
chunk_size = 1000

for start in range(0, len(images), chunk_size):

    chunk = images[start:start + chunk_size]

    # Convert pixel values:
    # 0...255  ->  0...1
    chunk = chunk.astype(np.float64) / 255.0

    channel_sum += chunk.sum(axis=(0, 1, 2))
    channel_squared_sum += (chunk ** 2).sum(axis=(0, 1, 2))

    total_pixels += (
        chunk.shape[0]
        * chunk.shape[1]
        * chunk.shape[2]
    )

mean = channel_sum / total_pixels

variance = (
    channel_squared_sum / total_pixels
    - mean ** 2
)

std = np.sqrt(variance)

print("\nCIFAR-100 mean:")
print(tuple(mean))

print("\nCIFAR-100 std:")
print(tuple(std))

## Step 3 — Create the CIFAR-100 Data Pipeline

The existing project already contains a CIFAR-10 data module.

To avoid changing the completed CIFAR-10 experiments, we create a separate:

`data/cifar100.py`

The CIFAR-100 data module performs:

1. CIFAR-100 downloading
2. Training augmentation
3. Validation/test preprocessing
4. A stratified 45,000 / 5,000 train-validation split
5. Training, validation, and test DataLoader creation

The split is stratified so that all 100 classes remain equally represented.

In [ ]:
from pathlib import Path

# Convert the values we calculated earlier into normal Python tuples.
CIFAR100_MEAN = tuple(float(x) for x in mean)
CIFAR100_STD = tuple(float(x) for x in std)

file_content = f'''import random

import numpy as np
import torch

from sklearn.model_selection import train_test_split

from torch.utils.data import (
    DataLoader,
    Subset,
)

from torchvision import (
    datasets,
    transforms,
)

from config import (
    DATA_DIR,
    NUM_WORKERS,
    PIN_MEMORY,
    SEED,
)


# ---------------------------------------------------------
# CIFAR-100 settings
# ---------------------------------------------------------

TRAIN_SIZE = 45_000
VAL_SIZE = 5_000
NUM_CLASSES = 100

CIFAR100_MEAN = {CIFAR100_MEAN!r}
CIFAR100_STD = {CIFAR100_STD!r}


def seed_worker(worker_id):
    """
    Give each DataLoader worker a reproducible random seed.
    """

    worker_seed = torch.initial_seed() % 2**32

    np.random.seed(worker_seed)
    random.seed(worker_seed)


class CIFAR100DataModule:
    """
    Handles CIFAR-100 downloading, preprocessing,
    stratified train/validation splitting,
    and DataLoader creation.
    """

    def __init__(
        self,
        data_dir=DATA_DIR,
    ):

        self.data_dir = data_dir

        self.train_dataset = None
        self.val_dataset = None
        self.test_dataset = None

        self.train_indices = None
        self.val_indices = None

        # Persistent generator used for reproducible
        # training-data shuffling.
        self.train_generator = torch.Generator()

        self.train_generator.manual_seed(SEED)

        self._prepare_datasets()


    def _prepare_datasets(self):
        """
        Download CIFAR-100 and create a fixed,
        stratified 45k / 5k train-validation split.
        """

        # -------------------------------------------------
        # Training transformations
        # -------------------------------------------------

        train_transform = transforms.Compose(
            [
                transforms.RandomCrop(
                    32,
                    padding=4,
                ),

                transforms.RandomHorizontalFlip(),

                transforms.ToTensor(),

                transforms.Normalize(
                    CIFAR100_MEAN,
                    CIFAR100_STD,
                ),
            ]
        )


        # -------------------------------------------------
        # Validation / test transformations
        # -------------------------------------------------

        evaluation_transform = transforms.Compose(
            [
                transforms.ToTensor(),

                transforms.Normalize(
                    CIFAR100_MEAN,
                    CIFAR100_STD,
                ),
            ]
        )


        # -------------------------------------------------
        # Load CIFAR-100
        # -------------------------------------------------

        # Augmented copy used for training.
        full_train_augmented = datasets.CIFAR100(
            root=self.data_dir,
            train=True,
            download=True,
            transform=train_transform,
        )


        # Non-augmented copy used for validation.
        full_train_evaluation = datasets.CIFAR100(
            root=self.data_dir,
            train=True,
            download=False,
            transform=evaluation_transform,
        )


        # Keep the official test set separate.
        self.test_dataset = datasets.CIFAR100(
            root=self.data_dir,
            train=False,
            download=True,
            transform=evaluation_transform,
        )


        # -------------------------------------------------
        # Stratified train / validation split
        # -------------------------------------------------

        targets = np.array(
            full_train_augmented.targets
        )

        all_indices = np.arange(
            len(targets)
        )


        (
            train_indices,
            val_indices,
        ) = train_test_split(

            all_indices,

            train_size=TRAIN_SIZE,

            test_size=VAL_SIZE,

            stratify=targets,

            random_state=SEED,

            shuffle=True,
        )


        self.train_indices = train_indices.tolist()
        self.val_indices = val_indices.tolist()


        # -------------------------------------------------
        # Build datasets
        # -------------------------------------------------

        self.train_dataset = Subset(
            full_train_augmented,
            self.train_indices,
        )

        self.val_dataset = Subset(
            full_train_evaluation,
            self.val_indices,
        )


        self._print_dataset_information(
            targets
        )


    def get_train_loader(
        self,
        batch_size,
    ):
        """
        Create a training DataLoader.

        batch_size is supplied dynamically so the
        adaptive batch controller can rebuild this
        loader when necessary.
        """

        return DataLoader(
            self.train_dataset,

            batch_size=batch_size,

            shuffle=True,

            num_workers=NUM_WORKERS,

            pin_memory=PIN_MEMORY,

            worker_init_fn=seed_worker,

            generator=self.train_generator,
        )


    def get_val_loader(
        self,
        batch_size=256,
    ):
        """
        Validation DataLoader.
        """

        return DataLoader(
            self.val_dataset,

            batch_size=batch_size,

            shuffle=False,

            num_workers=NUM_WORKERS,

            pin_memory=PIN_MEMORY,

            worker_init_fn=seed_worker,
        )


    def get_test_loader(
        self,
        batch_size=256,
    ):
        """
        Official CIFAR-100 test DataLoader.
        """

        return DataLoader(
            self.test_dataset,

            batch_size=batch_size,

            shuffle=False,

            num_workers=NUM_WORKERS,

            pin_memory=PIN_MEMORY,

            worker_init_fn=seed_worker,
        )


    def _print_dataset_information(
        self,
        targets,
    ):
        """
        Print dataset sizes and class distributions.
        """

        train_targets = targets[
            np.array(self.train_indices)
        ]

        val_targets = targets[
            np.array(self.val_indices)
        ]


        train_counts = np.bincount(
            train_targets,
            minlength=NUM_CLASSES,
        )

        val_counts = np.bincount(
            val_targets,
            minlength=NUM_CLASSES,
        )


        print("\\nCIFAR-100 dataset ready")
        print("-" * 60)

        print(
            f"Training samples:   {{len(self.train_dataset):,}}"
        )

        print(
            f"Validation samples: {{len(self.val_dataset):,}}"
        )

        print(
            f"Test samples:       {{len(self.test_dataset):,}}"
        )


        print("\\nTraining samples per class:")
        print(train_counts.tolist())

        print("\\nValidation samples per class:")
        print(val_counts.tolist())

        print("-" * 60)
'''

Path("data/cifar100.py").write_text(
    file_content,
    encoding="utf-8"
)

print("Created: data/cifar100.py")
print("CIFAR-100 mean:", CIFAR100_MEAN)
print("CIFAR-100 std: ", CIFAR100_STD)

## Step 4 — Verify the CIFAR-100 Train / Validation Split

We now instantiate the new `CIFAR100DataModule`.

Expected dataset sizes:

- Training: 45,000 images
- Validation: 5,000 images
- Test: 10,000 images

Because CIFAR-100 is balanced and the split is stratified, we expect:

- 450 training images per class
- 50 validation images per class

The official 10,000-image test set remains separate from training and validation.

In [ ]:
from data.cifar100 import CIFAR100DataModule

data100 = CIFAR100DataModule()

## Step 5 — Inspect One CIFAR-100 Training Batch

Before passing the data into the CNN, we verify that the DataLoader produces
tensors in the expected format.

For batch size 32, the expected image tensor shape is:

`[32, 3, 32, 32]`

where:

- 32 = number of images in the batch
- 3 = RGB channels
- 32 × 32 = image dimensions

The label tensor should contain one integer class ID for each image.
CIFAR-100 class IDs range from 0 to 99.

In [ ]:
train_loader100 = data100.get_train_loader(batch_size=32)

images, labels = next(iter(train_loader100))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)

print("Images dtype:", images.dtype)
print("Labels dtype:", labels.dtype)

print("First 10 labels:", labels[:10])
print("Min label:", labels.min().item())
print("Max label:", labels.max().item())

## Step 6 — Verify the CNN for 100 Classes

The existing `CustomCNN` has been designed so that the number of output classes
can be changed using the `num_classes` argument.

For CIFAR-100 we create:

`CustomCNN(num_classes=100)`

A batch containing 32 images should therefore produce an output tensor with shape:

`[32, 100]`

This means that the network produces 100 raw class scores (logits) for every
input image.

At this stage the model is untrained, so the predicted classes are not expected
to be meaningful.

In [ ]:
from models.custom_cnn import CustomCNN

model100 = CustomCNN(num_classes=100)

outputs = model100(images)

print("Input shape: ", images.shape)
print("Output shape:", outputs.shape)

print("\nFirst image output shape:")
print(outputs[0].shape)

print("\nPredicted classes for this batch:")
predictions = outputs.argmax(dim=1)
print(predictions)

print("\nPrediction min:", predictions.min().item())
print("Prediction max:", predictions.max().item())

## Step 7 — One-Epoch CIFAR-100 Training Smoke Test

Before running the full experiments, we perform a one-epoch smoke test.

The purpose of this test is NOT to obtain high classification accuracy.

Instead, it verifies that the complete pipeline works correctly:

CIFAR-100 DataLoader
→ CNN
→ CrossEntropyLoss
→ Backpropagation
→ SGD optimizer
→ Validation
→ Gradient statistics
→ Result logging
→ Checkpoint saving

For this smoke test we keep the same initial optimization settings used in the
CIFAR-10 study:

- Batch size: 32
- Learning rate: 0.01
- Optimizer: SGD
- Momentum: 0.9
- Weight decay: 5e-4

Only one epoch is run.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import time

from config import DEVICE, MOMENTUM, WEIGHT_DECAY, set_seed
from models.custom_cnn import CustomCNN
from training.trainer import Trainer


# ---------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------

set_seed(42)


# ---------------------------------------------------------
# Fresh CIFAR-100 model
# ---------------------------------------------------------

model100 = CustomCNN(
    num_classes=100
).to(DEVICE)


# ---------------------------------------------------------
# Loss
# ---------------------------------------------------------

criterion = nn.CrossEntropyLoss()


# ---------------------------------------------------------
# Optimizer
#
# Same frozen starting settings as CIFAR-10
# ---------------------------------------------------------

optimizer = optim.SGD(
    model100.parameters(),
    lr=0.01,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)


# ---------------------------------------------------------
# Data loaders
# ---------------------------------------------------------

train_loader100 = data100.get_train_loader(
    batch_size=32
)

val_loader100 = data100.get_val_loader(
    batch_size=256
)


# ---------------------------------------------------------
# Trainer
# ---------------------------------------------------------

trainer100 = Trainer(
    model=model100,
    criterion=criterion,
    optimizer=optimizer,
    device=DEVICE,
    results_dir="results/cifar100_smoke_test",
    checkpoint_dir="checkpoints/cifar100_smoke_test",
    run_name="cifar100_smoke_test",
    run_metadata={
        "dataset": "CIFAR-100",
        "experiment": "1_epoch_smoke_test",
        "num_classes": 100,
        "training_samples": 45000,
        "validation_samples": 5000,
        "batch_size": 32,
        "learning_rate": 0.01,
    },
)


# ---------------------------------------------------------
# Run ONE epoch only
# ---------------------------------------------------------

history100 = trainer100.fit(
    train_loader=train_loader100,
    val_loader=val_loader100,
    epochs=1,
    batch_size=32,
)

### Step 7 Result — Smoke Test Successful

The one-epoch CIFAR-100 smoke test completed successfully.

Observed results:

- Training loss: 4.1517
- Training accuracy: 5.96%
- Validation loss: 3.9896
- Validation accuracy: 7.76%
- Gradient-norm CV: 0.1546
- Learning rate: 0.01
- Batch size: 32
- Optimizer updates: 1,407
- Training time: 25.19 seconds

The run confirms that the CIFAR-100 data pipeline, 100-class CNN output,
loss function, optimizer, gradient tracking, validation loop, checkpointing,
and result logging work together without errors.

Random classification over 100 classes is approximately 1%, so the observed
training and validation accuracies are consistent with learning having begun.

This run is only a development smoke test and is NOT included in the final
E1–E4 experimental results.

## Step 8 — Save the Initial CIFAR-100 Pipeline to Git

The CIFAR-100 data pipeline and one-epoch smoke test have now been verified.

Before starting the full experiments, we inspect the Git working tree to make
sure that only the intended source-code changes are committed.

Experiment outputs, downloaded datasets, and checkpoints should not be
accidentally committed to the repository.

# E1 — CIFAR-100 Fixed Batch + Fixed Learning Rate

Baseline experiment using the same frozen optimization settings as the CIFAR-10 study.

- Batch size = 32
- Learning rate = 0.01
- Epochs = 20
- Seed = 42
- SGD, momentum = 0.9, weight decay = 5e-4

The test set is not used during this stage.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from config import (
    DEVICE,
    MOMENTUM,
    WEIGHT_DECAY,
    set_seed,
)

from data.cifar100 import CIFAR100DataModule
from models.custom_cnn import CustomCNN
from training.trainer import Trainer


# ---------------------------------------------------------
# E1 configuration
# ---------------------------------------------------------

E1_BATCH_SIZE = 32
E1_LEARNING_RATE = 0.01
E1_EPOCHS = 20
E1_SEED = 42

RUN_NAME = "cifar100_E1_fixed_fixed_batch32_lr0.01_seed42"


# ---------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------

set_seed(E1_SEED)


# ---------------------------------------------------------
# CIFAR-100 data
# ---------------------------------------------------------

data100_e1 = CIFAR100DataModule()

data100_e1.train_generator.manual_seed(E1_SEED)

train_loader_e1 = data100_e1.get_train_loader(
    batch_size=E1_BATCH_SIZE
)

val_loader_e1 = data100_e1.get_val_loader(
    batch_size=256
)


# ---------------------------------------------------------
# Fresh model
# ---------------------------------------------------------

model_e1 = CustomCNN(
    num_classes=100
).to(DEVICE)


# ---------------------------------------------------------
# Loss + optimizer
# ---------------------------------------------------------

criterion_e1 = nn.CrossEntropyLoss()

optimizer_e1 = optim.SGD(
    model_e1.parameters(),
    lr=E1_LEARNING_RATE,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)


# ---------------------------------------------------------
# Expected optimizer updates
# ---------------------------------------------------------

updates_per_epoch = len(train_loader_e1)
expected_total_updates = updates_per_epoch * E1_EPOCHS

print("E1 — CIFAR-100 Fixed Batch + Fixed LR")
print("=" * 60)
print(f"Device:             {DEVICE}")
print(f"Training samples:   {len(data100_e1.train_dataset):,}")
print(f"Validation samples: {len(data100_e1.val_dataset):,}")
print(f"Batch size:         {E1_BATCH_SIZE}")
print(f"Learning rate:      {E1_LEARNING_RATE}")
print(f"Epochs:             {E1_EPOCHS}")
print(f"Updates per epoch:  {updates_per_epoch:,}")
print(f"Expected updates:   {expected_total_updates:,}")
print("=" * 60)


# ---------------------------------------------------------
# Trainer
# ---------------------------------------------------------

trainer_e1 = Trainer(
    model=model_e1,
    criterion=criterion_e1,
    optimizer=optimizer_e1,
    device=DEVICE,
    results_dir="results/cifar100/E1",
    checkpoint_dir="checkpoints/cifar100/E1",
    run_name=RUN_NAME,
    run_metadata={
        "dataset": "CIFAR-100",
        "experiment": "E1_fixed_batch_fixed_lr",
        "num_classes": 100,
        "training_samples": 45000,
        "validation_samples": 5000,
        "test_set_used_during_training": False,
        "batch_size": E1_BATCH_SIZE,
        "learning_rate": E1_LEARNING_RATE,
        "epochs": E1_EPOCHS,
        "seed": E1_SEED,
        "optimizer": "SGD",
        "momentum": MOMENTUM,
        "weight_decay": WEIGHT_DECAY,
        "expected_total_updates": expected_total_updates,
    },
)


# ---------------------------------------------------------
# Full 20-epoch E1 run
# ---------------------------------------------------------

history_e1 = trainer_e1.fit(
    train_loader=train_loader_e1,
    val_loader=val_loader_e1,
    epochs=E1_EPOCHS,
    batch_size=E1_BATCH_SIZE,
)

### E1 Result

E1 completed successfully using a fixed batch size of 32 and fixed learning rate of 0.01.

- Best epoch: 17
- Best validation loss: 2.7953
- Best validation accuracy: 30.00%
- Total optimizer updates: 28,140
- Total training time: 444.38 s (~7.4 min)

The saved best checkpoint corresponds to epoch 17. The official CIFAR-100 test set has not been used.

In [ ]:
CIFAR100_CLASSES = tuple(
    data100_e1.test_dataset.classes
)

e1_evaluation = run_evaluation(
    model=model_e1,
    data_loader=val_loader_e1,
    device=DEVICE,
    history=history_e1,
    total_training_seconds=trainer_e1.total_training_seconds,
    checkpoint_path=(
        "checkpoints/cifar100/E1/"
        "cifar100_E1_fixed_fixed_batch32_lr0.01_seed42_best.pt"
    ),
    output_dir="results/cifar100/E1",
    run_name=RUN_NAME,
    split="validation",
    target_accuracy=0.30,
    class_names=CIFAR100_CLASSES,
    save_confusion_matrices=False,
)

### E1 Validation Evaluation

Best checkpoint: epoch 17

- Validation accuracy: 30.00%
- Validation macro F1: 0.2847
- Total optimizer updates: 28,140
- Training time: 444.38 seconds (~7.4 minutes)

The official CIFAR-100 test set remains untouched.

# E2 — CIFAR-100 Adaptive Batch + Fixed Learning Rate

Adaptive batch-size experiment using the frozen controller from the CIFAR-10 study.

- Initial batch size = 32
- Allowed batches = 32 → 64 → 128
- Learning rate = 0.01 fixed
- Epochs = 20
- Seed = 42
- Test set remains untouched

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from config import (
    DEVICE,
    MOMENTUM,
    WEIGHT_DECAY,
    set_seed,
)

from data.cifar100 import CIFAR100DataModule
from models.custom_cnn import CustomCNN
from training.batch_controller import AdaptiveBatchController
from training.trainer import Trainer


# ---------------------------------------------------------
# E2 configuration
# ---------------------------------------------------------

E2_INITIAL_BATCH = 32
E2_LEARNING_RATE = 0.01
E2_EPOCHS = 20
E2_SEED = 42

E2_BATCH_OPTIONS = [
    32,
    64,
    128,
]

CV_WINDOW = 3
STABILITY_THRESHOLD = 0.25
PLATEAU_PATIENCE = 3
MIN_DELTA = 0.01
COOLDOWN_EPOCHS = 2

RUN_NAME_E2 = "cifar100_E2_adaptive_batch_fixed_lr0.01_seed42"


# ---------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------

set_seed(E2_SEED)


# ---------------------------------------------------------
# CIFAR-100 data
# ---------------------------------------------------------

data100_e2 = CIFAR100DataModule()

data100_e2.train_generator.manual_seed(
    E2_SEED
)

train_loader_e2 = data100_e2.get_train_loader(
    batch_size=E2_INITIAL_BATCH
)

val_loader_e2 = data100_e2.get_val_loader(
    batch_size=256
)


# ---------------------------------------------------------
# Fresh model
# ---------------------------------------------------------

model_e2 = CustomCNN(
    num_classes=100
).to(DEVICE)


# ---------------------------------------------------------
# Loss + FIXED learning-rate optimizer
# ---------------------------------------------------------

criterion_e2 = nn.CrossEntropyLoss()

optimizer_e2 = optim.SGD(
    model_e2.parameters(),
    lr=E2_LEARNING_RATE,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)


# ---------------------------------------------------------
# Adaptive batch controller
# ---------------------------------------------------------

batch_controller_e2 = AdaptiveBatchController(
    batch_sizes=E2_BATCH_OPTIONS,
    initial_batch_size=E2_INITIAL_BATCH,
    cv_window=CV_WINDOW,
    stability_threshold=STABILITY_THRESHOLD,
    plateau_patience=PLATEAU_PATIENCE,
    min_delta=MIN_DELTA,
    cooldown_epochs=COOLDOWN_EPOCHS,
)


# ---------------------------------------------------------
# Experiment settings
# ---------------------------------------------------------

print("E2 — CIFAR-100 Adaptive Batch + Fixed LR")
print("=" * 60)
print(f"Device:                {DEVICE}")
print(f"Training samples:      {len(data100_e2.train_dataset):,}")
print(f"Validation samples:    {len(data100_e2.val_dataset):,}")
print(f"Initial batch size:    {E2_INITIAL_BATCH}")
print(f"Allowed batch sizes:   {E2_BATCH_OPTIONS}")
print(f"Fixed learning rate:   {E2_LEARNING_RATE}")
print(f"Epochs:                {E2_EPOCHS}")
print(f"CV window:             {CV_WINDOW}")
print(f"Stability threshold:   {STABILITY_THRESHOLD}")
print(f"Plateau patience:      {PLATEAU_PATIENCE}")
print(f"Minimum delta:         {MIN_DELTA}")
print(f"Cooldown epochs:       {COOLDOWN_EPOCHS}")
print("=" * 60)


# ---------------------------------------------------------
# Trainer
# ---------------------------------------------------------

trainer_e2 = Trainer(
    model=model_e2,
    criterion=criterion_e2,
    optimizer=optimizer_e2,
    device=DEVICE,
    results_dir="results/cifar100/E2",
    checkpoint_dir="checkpoints/cifar100/E2",
    run_name=RUN_NAME_E2,
    run_metadata={
        "dataset": "CIFAR-100",
        "experiment": "E2_adaptive_batch_fixed_lr",
        "num_classes": 100,
        "training_samples": 45000,
        "validation_samples": 5000,
        "test_set_used_during_training": False,
        "initial_batch_size": E2_INITIAL_BATCH,
        "allowed_batch_sizes": E2_BATCH_OPTIONS,
        "learning_rate": E2_LEARNING_RATE,
        "learning_rate_policy": "fixed",
        "epochs": E2_EPOCHS,
        "seed": E2_SEED,
        "optimizer": "SGD",
        "momentum": MOMENTUM,
        "weight_decay": WEIGHT_DECAY,
        "cv_window": CV_WINDOW,
        "stability_threshold": STABILITY_THRESHOLD,
        "plateau_patience": PLATEAU_PATIENCE,
        "min_delta": MIN_DELTA,
        "cooldown_epochs": COOLDOWN_EPOCHS,
    },
)


# ---------------------------------------------------------
# Full E2 run
# ---------------------------------------------------------

history_e2 = trainer_e2.fit(
    train_loader=train_loader_e2,
    val_loader=val_loader_e2,
    epochs=E2_EPOCHS,
    batch_size=E2_INITIAL_BATCH,
    batch_controller=batch_controller_e2,
    train_loader_factory=data100_e2.get_train_loader,
)

### E2 Result

The adaptive batch controller increased the batch size from 32 to 64 after
epoch 13 when both the validation-loss plateau and gradient-stability
conditions were satisfied.

- Best checkpoint: epoch 17
- Best validation loss: 2.7228
- Validation accuracy at best checkpoint: 30.96%
- Total optimizer updates: 23,219
- Training time: 436.16 seconds (~7.3 minutes)
- Batch sizes actually used: 32 → 64
- A change to batch 128 was triggered after epoch 20, but no training epoch
  was performed at batch 128.
- Learning rate remained fixed at 0.01.

Compared with E1, E2 used approximately 17.5% fewer optimizer updates.
The official CIFAR-100 test set remains untouched.



In [ ]:
e2_evaluation = run_evaluation(
    model=model_e2,
    data_loader=val_loader_e2,
    device=DEVICE,
    history=history_e2,
    total_training_seconds=trainer_e2.total_training_seconds,
    checkpoint_path=(
        "checkpoints/cifar100/E2/"
        "cifar100_E2_adaptive_batch_fixed_lr0.01_seed42_best.pt"
    ),
    output_dir="results/cifar100/E2",
    run_name=RUN_NAME_E2,
    split="validation",
    target_accuracy=0.30,
    class_names=CIFAR100_CLASSES,
    save_confusion_matrices=False,
)

### E2 Validation Evaluation

The adaptive batch controller increased the batch size from 32 to 64 after
epoch 13 when the validation-loss plateau and gradient-stability conditions
were both satisfied.

- Best checkpoint: epoch 17
- Best validation loss: 2.7228
- Validation accuracy: 30.96%
- Validation macro F1: 0.2890
- Total optimizer updates: 23,219
- Training time: 436.16 seconds (~7.3 minutes)
- Batch sizes actually used: 32 → 64
- Learning rate remained fixed at 0.01
- A transition to batch 128 was triggered after epoch 20, but batch 128 was
  never used for a training epoch.

The official CIFAR-100 test set remains untouched.

# E3 — CIFAR-100 Fixed Batch + Adaptive Learning Rate

Adaptive learning-rate experiment using the frozen LR controller from the
CIFAR-10 study.

- Batch size = 32 fixed
- Reference learning rate = 0.01
- Adaptive LR decay factor = 0.5
- Epochs = 20
- Seed = 42
- Test set remains untouched

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from config import (
    DEVICE,
    MOMENTUM,
    WEIGHT_DECAY,
    set_seed,
)

from data.cifar100 import CIFAR100DataModule
from models.custom_cnn import CustomCNN
from training.lr_controller import LRController
from training.trainer import Trainer


# ---------------------------------------------------------
# E3 configuration
# ---------------------------------------------------------

E3_BATCH_SIZE = 32
E3_REFERENCE_LR = 0.01
E3_EPOCHS = 20
E3_SEED = 42

E3_REFERENCE_BATCH = 32
E3_ALPHA = 0.5
E3_LR_FACTOR = 0.5
E3_PLATEAU_PATIENCE = 5
E3_WORSENING_PATIENCE = 3
E3_COOLDOWN_EPOCHS = 2
E3_WARMUP_EPOCHS = 0
E3_MIN_LR = 1e-5
E3_MIN_DELTA_LOSS = 1e-3
E3_MIN_DELTA_ACC = 0.002

RUN_NAME_E3 = "cifar100_E3_fixed_batch_adaptive_lr_seed42"


# ---------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------

set_seed(E3_SEED)


# ---------------------------------------------------------
# CIFAR-100 data
# ---------------------------------------------------------

data100_e3 = CIFAR100DataModule()

data100_e3.train_generator.manual_seed(
    E3_SEED
)

train_loader_e3 = data100_e3.get_train_loader(
    batch_size=E3_BATCH_SIZE
)

val_loader_e3 = data100_e3.get_val_loader(
    batch_size=256
)


# ---------------------------------------------------------
# Fresh model
# ---------------------------------------------------------

model_e3 = CustomCNN(
    num_classes=100
).to(DEVICE)


# ---------------------------------------------------------
# Optimizer
# ---------------------------------------------------------

criterion_e3 = nn.CrossEntropyLoss()

optimizer_e3 = optim.SGD(
    model_e3.parameters(),
    lr=E3_REFERENCE_LR,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)


# ---------------------------------------------------------
# Adaptive LR controller
# ---------------------------------------------------------

lr_controller_e3 = LRController(
    optimizer_e3,
    mode="adaptive",
    reference_batch_size=E3_REFERENCE_BATCH,
    reference_lr=E3_REFERENCE_LR,
    alpha=E3_ALPHA,
    lr_factor=E3_LR_FACTOR,
    plateau_patience=E3_PLATEAU_PATIENCE,
    worsening_patience=E3_WORSENING_PATIENCE,
    cooldown_epochs=E3_COOLDOWN_EPOCHS,
    warmup_epochs=E3_WARMUP_EPOCHS,
    min_lr=E3_MIN_LR,
    min_delta_loss=E3_MIN_DELTA_LOSS,
    min_delta_acc=E3_MIN_DELTA_ACC,
)

initial_lr_e3 = lr_controller_e3.set_batch_size(
    E3_BATCH_SIZE
)


# ---------------------------------------------------------
# Experiment settings
# ---------------------------------------------------------

print("E3 — CIFAR-100 Fixed Batch + Adaptive LR")
print("=" * 60)
print(f"Device:                 {DEVICE}")
print(f"Training samples:       {len(data100_e3.train_dataset):,}")
print(f"Validation samples:     {len(data100_e3.val_dataset):,}")
print(f"Fixed batch size:       {E3_BATCH_SIZE}")
print(f"Initial learning rate:  {initial_lr_e3}")
print(f"Reference LR:           {E3_REFERENCE_LR}")
print(f"Reference batch:        {E3_REFERENCE_BATCH}")
print(f"LR scaling alpha:       {E3_ALPHA}")
print(f"LR decay factor:        {E3_LR_FACTOR}")
print(f"Plateau patience:       {E3_PLATEAU_PATIENCE}")
print(f"Worsening patience:     {E3_WORSENING_PATIENCE}")
print(f"Cooldown epochs:        {E3_COOLDOWN_EPOCHS}")
print(f"Minimum LR:             {E3_MIN_LR}")
print(f"Epochs:                 {E3_EPOCHS}")
print("=" * 60)


# ---------------------------------------------------------
# Trainer
# ---------------------------------------------------------

trainer_e3 = Trainer(
    model=model_e3,
    criterion=criterion_e3,
    optimizer=optimizer_e3,
    device=DEVICE,
    results_dir="results/cifar100/E3",
    checkpoint_dir="checkpoints/cifar100/E3",
    run_name=RUN_NAME_E3,
    run_metadata={
        "dataset": "CIFAR-100",
        "experiment": "E3_fixed_batch_adaptive_lr",
        "num_classes": 100,
        "training_samples": 45000,
        "validation_samples": 5000,
        "test_set_used_during_training": False,
        "batch_size": E3_BATCH_SIZE,
        "reference_learning_rate": E3_REFERENCE_LR,
        "reference_batch_size": E3_REFERENCE_BATCH,
        "alpha": E3_ALPHA,
        "lr_factor": E3_LR_FACTOR,
        "plateau_patience": E3_PLATEAU_PATIENCE,
        "worsening_patience": E3_WORSENING_PATIENCE,
        "cooldown_epochs": E3_COOLDOWN_EPOCHS,
        "warmup_epochs": E3_WARMUP_EPOCHS,
        "min_lr": E3_MIN_LR,
        "min_delta_loss": E3_MIN_DELTA_LOSS,
        "min_delta_acc": E3_MIN_DELTA_ACC,
        "epochs": E3_EPOCHS,
        "seed": E3_SEED,
        "optimizer": "SGD",
        "momentum": MOMENTUM,
        "weight_decay": WEIGHT_DECAY,
    },
)


# ---------------------------------------------------------
# Full E3 run
# ---------------------------------------------------------

history_e3 = trainer_e3.fit(
    train_loader=train_loader_e3,
    val_loader=val_loader_e3,
    epochs=E3_EPOCHS,
    batch_size=E3_BATCH_SIZE,
    lr_controller=lr_controller_e3,
)

### E3 Result

The adaptive learning-rate controller kept the initial learning rate of 0.01
through epoch 13.

After epoch 13, worsening validation behavior triggered a learning-rate decay:

- 0.0100 → 0.0050

A second decay was triggered after epoch 19:

- 0.0050 → 0.0025

Final results:

- Best checkpoint: epoch 20
- Best validation loss: 2.6273
- Validation accuracy: 33.32%
- Total optimizer updates: 28,140
- Training time: 457.03 seconds (~7.6 minutes)
- Batch size remained fixed at 32
- Final learning rate used: 0.0025

The official CIFAR-100 test set remains untouched.

In [ ]:
e3_evaluation = run_evaluation(
    model=model_e3,
    data_loader=val_loader_e3,
    device=DEVICE,
    history=history_e3,
    total_training_seconds=trainer_e3.total_training_seconds,
    checkpoint_path=(
        "checkpoints/cifar100/E3/"
        "cifar100_E3_fixed_batch_adaptive_lr_seed42_best.pt"
    ),
    output_dir="results/cifar100/E3",
    run_name=RUN_NAME_E3,
    split="validation",
    target_accuracy=0.30,
    class_names=CIFAR100_CLASSES,
    save_confusion_matrices=False,
)

# E4 — CIFAR-100 Adaptive Batch + Adaptive Learning Rate

Combined adaptive experiment using both frozen controllers from the CIFAR-10
study.

- Initial batch size = 32
- Allowed batch sizes = 32 → 64 → 128
- Reference learning rate = 0.01
- Adaptive LR scaling and decay enabled
- Epochs = 20
- Seed = 42
- Test set remains untouched

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from config import (
    DEVICE,
    MOMENTUM,
    WEIGHT_DECAY,
    set_seed,
)

from data.cifar100 import CIFAR100DataModule
from models.custom_cnn import CustomCNN
from training.batch_controller import AdaptiveBatchController
from training.lr_controller import LRController
from training.trainer import Trainer


# ---------------------------------------------------------
# E4 configuration
# ---------------------------------------------------------

E4_INITIAL_BATCH = 32
E4_BATCH_OPTIONS = [32, 64, 128]

E4_REFERENCE_LR = 0.01
E4_REFERENCE_BATCH = 32
E4_ALPHA = 0.5

E4_EPOCHS = 20
E4_SEED = 42

# Batch controller
E4_CV_WINDOW = 3
E4_STABILITY_THRESHOLD = 0.25
E4_BATCH_PLATEAU_PATIENCE = 3
E4_BATCH_MIN_DELTA = 0.01
E4_BATCH_COOLDOWN = 2

# LR controller
E4_LR_FACTOR = 0.5
E4_LR_PLATEAU_PATIENCE = 5
E4_WORSENING_PATIENCE = 3
E4_LR_COOLDOWN = 2
E4_WARMUP = 0
E4_MIN_LR = 1e-5
E4_MIN_DELTA_LOSS = 1e-3
E4_MIN_DELTA_ACC = 0.002

RUN_NAME_E4 = "cifar100_E4_adaptive_batch_adaptive_lr_seed42"


# ---------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------

set_seed(E4_SEED)


# ---------------------------------------------------------
# CIFAR-100 data
# ---------------------------------------------------------

data100_e4 = CIFAR100DataModule()

data100_e4.train_generator.manual_seed(E4_SEED)

train_loader_e4 = data100_e4.get_train_loader(
    batch_size=E4_INITIAL_BATCH
)

val_loader_e4 = data100_e4.get_val_loader(
    batch_size=256
)


# ---------------------------------------------------------
# Fresh model
# ---------------------------------------------------------

model_e4 = CustomCNN(
    num_classes=100
).to(DEVICE)


# ---------------------------------------------------------
# Optimizer
# ---------------------------------------------------------

criterion_e4 = nn.CrossEntropyLoss()

optimizer_e4 = optim.SGD(
    model_e4.parameters(),
    lr=E4_REFERENCE_LR,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)


# ---------------------------------------------------------
# Adaptive batch controller
# ---------------------------------------------------------

batch_controller_e4 = AdaptiveBatchController(
    batch_sizes=E4_BATCH_OPTIONS,
    initial_batch_size=E4_INITIAL_BATCH,
    cv_window=E4_CV_WINDOW,
    stability_threshold=E4_STABILITY_THRESHOLD,
    plateau_patience=E4_BATCH_PLATEAU_PATIENCE,
    min_delta=E4_BATCH_MIN_DELTA,
    cooldown_epochs=E4_BATCH_COOLDOWN,
)


# ---------------------------------------------------------
# Adaptive LR controller
# ---------------------------------------------------------

lr_controller_e4 = LRController(
    optimizer_e4,
    mode="adaptive",
    reference_batch_size=E4_REFERENCE_BATCH,
    reference_lr=E4_REFERENCE_LR,
    alpha=E4_ALPHA,
    lr_factor=E4_LR_FACTOR,
    plateau_patience=E4_LR_PLATEAU_PATIENCE,
    worsening_patience=E4_WORSENING_PATIENCE,
    cooldown_epochs=E4_LR_COOLDOWN,
    warmup_epochs=E4_WARMUP,
    min_lr=E4_MIN_LR,
    min_delta_loss=E4_MIN_DELTA_LOSS,
    min_delta_acc=E4_MIN_DELTA_ACC,
)

initial_lr_e4 = lr_controller_e4.set_batch_size(
    E4_INITIAL_BATCH
)


# ---------------------------------------------------------
# Experiment settings
# ---------------------------------------------------------

print("E4 — CIFAR-100 Adaptive Batch + Adaptive LR")
print("=" * 60)
print(f"Device:                 {DEVICE}")
print(f"Training samples:       {len(data100_e4.train_dataset):,}")
print(f"Validation samples:     {len(data100_e4.val_dataset):,}")
print(f"Initial batch size:     {E4_INITIAL_BATCH}")
print(f"Allowed batch sizes:    {E4_BATCH_OPTIONS}")
print(f"Initial learning rate:  {initial_lr_e4}")
print(f"Reference LR:           {E4_REFERENCE_LR}")
print(f"LR scaling alpha:       {E4_ALPHA}")
print(f"Epochs:                 {E4_EPOCHS}")
print("=" * 60)


# ---------------------------------------------------------
# Trainer
# ---------------------------------------------------------

trainer_e4 = Trainer(
    model=model_e4,
    criterion=criterion_e4,
    optimizer=optimizer_e4,
    device=DEVICE,
    results_dir="results/cifar100/E4",
    checkpoint_dir="checkpoints/cifar100/E4",
    run_name=RUN_NAME_E4,
    run_metadata={
        "dataset": "CIFAR-100",
        "experiment": "E4_adaptive_batch_adaptive_lr",
        "num_classes": 100,
        "training_samples": 45000,
        "validation_samples": 5000,
        "test_set_used_during_training": False,
        "initial_batch_size": E4_INITIAL_BATCH,
        "allowed_batch_sizes": E4_BATCH_OPTIONS,
        "reference_learning_rate": E4_REFERENCE_LR,
        "reference_batch_size": E4_REFERENCE_BATCH,
        "alpha": E4_ALPHA,
        "epochs": E4_EPOCHS,
        "seed": E4_SEED,
        "optimizer": "SGD",
        "momentum": MOMENTUM,
        "weight_decay": WEIGHT_DECAY,
    },
)


# ---------------------------------------------------------
# Full E4 run
# ---------------------------------------------------------

history_e4 = trainer_e4.fit(
    train_loader=train_loader_e4,
    val_loader=val_loader_e4,
    epochs=E4_EPOCHS,
    batch_size=E4_INITIAL_BATCH,
    batch_controller=batch_controller_e4,
    train_loader_factory=data100_e4.get_train_loader,
    lr_controller=lr_controller_e4,
)

### E4 Result

The combined controller increased the batch size from 32 to 64 after epoch 13.

At the same point, the adaptive LR controller triggered a decay. Because the
new batch size was 64, batch-based LR scaling and the decay factor interacted,
resulting in a next learning rate of 0.007071.

- Best checkpoint: epoch 17
- Best validation loss: 2.6809
- Validation accuracy: 31.82%
- Total optimizer updates: 23,219
- Training time: 442.02 seconds (~7.4 minutes)
- Batch sizes actually used: 32 → 64
- Learning rates actually used: 0.0100 → 0.007071
- Batch 128 and LR 0.005 were selected after epoch 20 but were never used for
  another training epoch.

The official CIFAR-100 test set remains untouched.

In [ ]:
e4_evaluation = run_evaluation(
    model=model_e4,
    data_loader=val_loader_e4,
    device=DEVICE,
    history=history_e4,
    total_training_seconds=trainer_e4.total_training_seconds,
    checkpoint_path=(
        "checkpoints/cifar100/E4/"
        "cifar100_E4_adaptive_batch_adaptive_lr_seed42_best.pt"
    ),
    output_dir="results/cifar100/E4",
    run_name=RUN_NAME_E4,
    split="validation",
    target_accuracy=0.30,
    class_names=CIFAR100_CLASSES,
    save_confusion_matrices=False,
)

### E4 Validation Evaluation

- Best checkpoint: epoch 17
- Best validation loss: 2.6809
- Validation accuracy: 31.82%
- Validation macro F1: 0.2974
- Total optimizer updates: 23,219
- Training time: 442.02 seconds (~7.4 minutes)
- Batch sizes actually used: 32 → 64
- Learning rates actually used: 0.0100 → 0.007071

A transition to batch 128 and LR 0.005 was selected after epoch 20,
but those settings were not used for another training epoch.

The official CIFAR-100 test set remains untouched.